# Selective prediction: PRR and ensemble agreement

Loads the per-example arrays written by `evaluation/selective_prediction.py`
(`results/selective_prediction/<dataset>_<model>.npz`) and reduces them to four tables,
one per quantity: **EU**, **AU**, **TU** and **MSE**. Each has the datasets down the
left, split into one row per method, and reports

* **PRR** -- the prediction-rejection ratio of that uncertainty over retention in
  `[0.5, 1]`. `1` ranks the test set as well as the realised error itself, `0` is no
  better than rejecting at random, negative is worse than random.
* **rho** -- Spearman rank correlation of that uncertainty against the **deep
  ensemble's** on the same test examples. Only the ranking matters for selective
  prediction, so rank correlation is the right lens. Undefined for the ensemble itself
  (it would be its own reference) and wherever the ensemble is missing.

The four methods are the three distributional heads -- diagonal Normal, Normal mixture,
multivariate Normal -- and the deterministic diffusion aggregated over five checkpoints.
All of them report AU and EU as variances on the same scale, so `tu = au + eu` holds and
the rows are directly comparable.

Generate the inputs first, e.g.

```bash
python evaluation/selective_prediction.py --datasets KS,Burgers,T2M
python evaluation/selective_prediction.py --datasets UCI_concrete,UCI_energy
```

Datasets with no results at all are skipped; a method missing for a dataset that does
have results shows `--`. T2M has only one deterministic checkpoint, so it has no
ensemble; the UCI targets are scalars, so no multivariate-Normal model was trained
there.

In [32]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, str(Path.cwd()))

# Reuse the retention/PRR definitions from the script so the two never drift apart.
import selective_prediction as sp

RESULTS_DIR = Path("../results/selective_prediction")

# Every dataset the script knows about, so the two never drift apart either.
DATASETS = list(sp.DATASETS)

# The model kinds and their table labels, declared here rather than read off
# sp.MODEL_KINDS: that tuple gets narrowed while a partial sweep is running, and the
# table should keep its full set of rows regardless.
MODELS = ["normal", "mixednormal", "mvnormal", "ensemble"]
MODEL_LABELS = {
    "normal": "Diagonal",
    "mixednormal": "Mixture",
    "mvnormal": "Multivariate",
    "ensemble": "Ensemble",
}
REFERENCE_MODEL = "ensemble"

# EU reduction variant to load: "" is the default (mean over all diffusion steps),
# other runs are suffixed e.g. "_max" or "_last5". The suffix applies to the
# distributional heads only -- the ensemble EU has no diffusion-step axis.
VARIANT = ""

# npz key -> the label used in the column headers and the curve build_curves returns.
MEASURES = {
    "eu": ("EU", "Epistemic uncertainty"),
    "au": ("AU", "Aleatoric uncertainty"),
    "tu": ("TU", "Total uncertainty"),
}

In [33]:
# results is keyed by (dataset, model). Missing files are reported and skipped, so
# the tables still build after a partial sweep.
results = {}
missing = []
for name in DATASETS:
    for model in MODELS:
        suffix = VARIANT if model != REFERENCE_MODEL else ""
        path = RESULTS_DIR / f"{name}_{model}{suffix}.npz"
        if not path.exists():
            missing.append(f"{name}/{model}")
            continue
        with np.load(path, allow_pickle=True) as f:
            results[(name, model)] = {k: f[k] for k in f.files}

print(f"loaded {len(results)} result files")
if missing:
    print(f"missing ({len(missing)}) -- run selective_prediction.py for them:")
    print("  " + ", ".join(missing))

loaded 33 result files
missing (11) -- run selective_prediction.py for them:
  T2M/mixednormal, T2M/mvnormal, T2M/ensemble, UCI_concrete/mvnormal, UCI_energy/mvnormal, UCI_kin8nm/mvnormal, UCI_naval/mvnormal, UCI_power/mvnormal, UCI_protein/mvnormal, UCI_wine/mvnormal, UCI_yacht/mvnormal


In [34]:
def same_examples(a, b):
    """True if two runs evaluated the same test examples, in the same order.

    ``target_mean`` is the per-example domain mean of the target, which does not
    depend on the model -- so it identifies the example without needing an index.
    The PDE test sets draw a random timestep per example, so two runs only line up
    when they were produced with the same --seed and --n-samples; correlating them
    otherwise would silently pair up different examples.
    """
    if "target_mean" not in a or "target_mean" not in b:
        return len(a["mse"]) == len(b["mse"])
    return a["target_mean"].shape == b["target_mean"].shape and np.allclose(
        a["target_mean"], b["target_mean"], rtol=0, atol=1e-6
    )


def row_stats(r, reference):
    """PRR and ensemble rank correlation per measure, plus the mean MSE, for one run."""
    grid = r["retention_grid"]
    curves, oracle, random_curve = sp.build_curves(r["au"], r["eu"], r["mse"], grid)

    stats = {"MSE": float(r["mse"].mean())}
    for key, (label, curve_name) in MEASURES.items():
        stats[f"PRR ({label})"] = sp.prediction_rejection_ratio(
            curves[curve_name], oracle, random_curve, grid
        )
        stats[f"rho ({label})"] = (
            np.nan if reference is None else spearmanr(r[key], reference[key])[0]
        )
    return stats


# A dataset whose sweep has not produced anything yet is left out entirely rather
# than filling the table with a block of dashes.
DATASETS_PRESENT = [
    name for name in DATASETS if any((name, model) in results for model in MODELS)
]

rows = {}
for name in DATASETS_PRESENT:
    ens = results.get((name, REFERENCE_MODEL))
    for model in MODELS:
        r = results.get((name, model))
        if r is None:
            continue
        # The ensemble is the reference, so it has no correlation against itself,
        # and neither has a run that cannot be lined up with it example by example.
        reference = None if model == REFERENCE_MODEL or ens is None else ens
        if reference is not None and not same_examples(r, reference):
            print(
                f"[{name}/{model}] did not evaluate the same examples as the "
                "ensemble -- re-run both with the same --seed and --n-samples. "
                "Correlations dropped."
            )
            reference = None
        rows[(name, MODEL_LABELS[model])] = row_stats(r, reference)

stats = pd.DataFrame.from_dict(rows, orient="index")
stats.index = pd.MultiIndex.from_tuples(stats.index, names=["Dataset", "Method"])

# Reindex onto the full grid: every dataset that has any result keeps all four
# method rows, so a method still running shows as -- rather than vanishing and
# silently shortening that dataset's block.
stats = stats.reindex(
    pd.MultiIndex.from_product(
        [DATASETS_PRESENT, [MODEL_LABELS[m] for m in MODELS]],
        names=["Dataset", "Method"],
    )
)


def measure_table(label):
    """The (PRR, rho) table for one measure, as displayed and as LaTeX."""
    table = stats.reindex(columns=[f"PRR ({label})", f"rho ({label})"])
    print(
        table.rename(columns={f"rho ({label})": rf"$\rho$ ({label})"}).to_latex(
            float_format="%.4f",
            na_rep="--",
            caption=f"PRR of the {label} and its rank correlation with the deep "
            "ensemble, per dataset and method.",
        )
    )
    return table.style.format("{:.4f}", na_rep="--")

## Epistemic uncertainty

PRR of the EU, and how closely each head's EU reproduces the deep ensemble's *ordering*
of the test examples.

In [35]:
measure_table("EU")

\begin{table}
\caption{PRR of the EU and its rank correlation with the deep ensemble, per dataset and method.}
\begin{tabular}{llrr}
\toprule
 &  & PRR (EU) & $\rho$ (EU) \\
Dataset & Method &  &  \\
\midrule
\multirow[t]{4}{*}{KS} & Diagonal & 0.8453 & 0.8210 \\
 & Mixture & 0.8135 & 0.7527 \\
 & Multivariate & 0.4088 & 0.5231 \\
 & Ensemble & 0.8177 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{Burgers} & Diagonal & 0.9945 & 0.8899 \\
 & Mixture & 0.9922 & 0.8887 \\
 & Multivariate & 0.9708 & 0.7710 \\
 & Ensemble & 0.9942 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{T2M} & Diagonal & 0.6951 & -- \\
 & Mixture & -- & -- \\
 & Multivariate & -- & -- \\
 & Ensemble & -- & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_concrete} & Diagonal & 0.3803 & 0.1510 \\
 & Mixture & 0.2622 & 0.1771 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.0939 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_energy} & Diagonal & 0.4161 & 0.5723 \\
 & Mixture & 0.4263 & 0.5164 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.0420 & 

## Aleatoric uncertainty

In [36]:
measure_table("AU")

\begin{table}
\caption{PRR of the AU and its rank correlation with the deep ensemble, per dataset and method.}
\begin{tabular}{llrr}
\toprule
 &  & PRR (AU) & $\rho$ (AU) \\
Dataset & Method &  &  \\
\midrule
\multirow[t]{4}{*}{KS} & Diagonal & 0.8394 & 0.9640 \\
 & Mixture & 0.8458 & 0.9297 \\
 & Multivariate & 0.2081 & 0.1792 \\
 & Ensemble & 0.6881 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{Burgers} & Diagonal & 0.9945 & 0.8185 \\
 & Mixture & 0.9927 & 0.8383 \\
 & Multivariate & 0.9453 & 0.2784 \\
 & Ensemble & 0.9899 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{T2M} & Diagonal & 0.7286 & -- \\
 & Mixture & -- & -- \\
 & Multivariate & -- & -- \\
 & Ensemble & -- & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_concrete} & Diagonal & 0.3903 & 0.8100 \\
 & Mixture & 0.2426 & 0.8554 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.2197 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_energy} & Diagonal & 0.5015 & 0.7664 \\
 & Mixture & 0.3769 & 0.7798 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.2224 & 

## Total uncertainty (`tu = au + eu`)

In [37]:
measure_table("TU")

\begin{table}
\caption{PRR of the TU and its rank correlation with the deep ensemble, per dataset and method.}
\begin{tabular}{llrr}
\toprule
 &  & PRR (TU) & $\rho$ (TU) \\
Dataset & Method &  &  \\
\midrule
\multirow[t]{4}{*}{KS} & Diagonal & 0.8402 & 0.9207 \\
 & Mixture & 0.8463 & 0.8839 \\
 & Multivariate & 0.2089 & 0.1910 \\
 & Ensemble & 0.7937 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{Burgers} & Diagonal & 0.9945 & 0.8839 \\
 & Mixture & 0.9927 & 0.8841 \\
 & Multivariate & 0.9456 & 0.2812 \\
 & Ensemble & 0.9921 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{T2M} & Diagonal & 0.7286 & -- \\
 & Mixture & -- & -- \\
 & Multivariate & -- & -- \\
 & Ensemble & -- & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_concrete} & Diagonal & 0.3953 & 0.7601 \\
 & Mixture & 0.2411 & 0.8020 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.2198 & -- \\
\cline{1-4}
\multirow[t]{4}{*}{UCI_energy} & Diagonal & 0.4986 & 0.6671 \\
 & Mixture & 0.3797 & 0.6515 \\
 & Multivariate & -- & -- \\
 & Ensemble & 0.0820 & 

## MSE

The squared error of each model's mean prediction against the target, averaged over the
domain and over the test set. In standardised units throughout, so it is comparable
across methods within a dataset but not between datasets.

In [38]:
mse_table = stats.reindex(columns=["MSE"])
print(
    mse_table.to_latex(
        float_format="%.2e",
        na_rep="--",
        caption="Mean MSE per dataset and method, in standardised units.",
    )
)
mse_table.style.format("{:.3e}", na_rep="--")

\begin{table}
\caption{Mean MSE per dataset and method, in standardised units.}
\begin{tabular}{llr}
\toprule
 &  & MSE \\
Dataset & Method &  \\
\midrule
\multirow[t]{4}{*}{KS} & Diagonal & 2.22e-06 \\
 & Mixture & 1.67e-06 \\
 & Multivariate & 3.36e-06 \\
 & Ensemble & 2.47e-06 \\
\cline{1-3}
\multirow[t]{4}{*}{Burgers} & Diagonal & 3.84e-06 \\
 & Mixture & 6.27e-06 \\
 & Multivariate & 3.84e-06 \\
 & Ensemble & 4.63e-06 \\
\cline{1-3}
\multirow[t]{4}{*}{T2M} & Diagonal & 6.12e-03 \\
 & Mixture & -- \\
 & Multivariate & -- \\
 & Ensemble & -- \\
\cline{1-3}
\multirow[t]{4}{*}{UCI_concrete} & Diagonal & 7.26e-02 \\
 & Mixture & 7.09e-02 \\
 & Multivariate & -- \\
 & Ensemble & 7.34e-02 \\
\cline{1-3}
\multirow[t]{4}{*}{UCI_energy} & Diagonal & 1.22e-03 \\
 & Mixture & 1.23e-03 \\
 & Multivariate & -- \\
 & Ensemble & 1.17e-03 \\
\cline{1-3}
\multirow[t]{4}{*}{UCI_kin8nm} & Diagonal & 6.92e-02 \\
 & Mixture & 6.86e-02 \\
 & Multivariate & -- \\
 & Ensemble & 6.91e-02 \\
\cline{1-3}
\mu